In [ ]:
import sys
from pathlib import Path

start = Path.cwd().resolve()
topic_dir = None
for directory in (start, *start.parents):
    if (directory / "find_stack_files.py").is_file() and (directory / "config.yaml").is_file():
        topic_dir = directory
        break
if topic_dir is None:
    raise FileNotFoundError(
        "Topic folder not found. Open the notebook from topics/11-B-FLOODED-FOREST "
        "or set the kernel working directory there."
    )
if str(topic_dir) not in sys.path:
    sys.path.insert(0, str(topic_dir))

import find_stack_files as fs

fs.init_settings()

CREDENTIALS_PATH = fs.CREDENTIALS_PATH
OUTPUT_JSON_PATH = fs.OUTPUT_JSON_PATH
load_credentials = fs.load_credentials
get_maap_access_token = fs.get_maap_access_token
main = fs.main

print("Topic dir:", topic_dir)
print("Credentials:", CREDENTIALS_PATH)
print("Output JSON:", OUTPUT_JSON_PATH)


In [ ]:
# Query MAAP STAC for all ROIs in rois.json (uses paths from config.yaml)
main()


In [ ]:
import json
import urllib.request
from io import BytesIO
from pathlib import Path

import matplotlib.pyplot as plt
from PIL import Image

json_path = Path(OUTPUT_JSON_PATH)
if not json_path.is_file():
    raise FileNotFoundError(
        f"{json_path} not found. Run the STAC search cell above first (main())."
    )

with json_path.open("r", encoding="utf-8") as f:
    data = json.load(f)

offline_token, client_id, client_secret = load_credentials(CREDENTIALS_PATH)
access_token = get_maap_access_token(offline_token, client_id, client_secret)

print(f"Loaded: {json_path.resolve()}")
print(f"Collection: {data.get('collection')}")
print("Products per ROI:")
for roi in data.get("rois", []):
    print(f"  {roi['name']}: {roi['product_count']}")


In [ ]:
# Region to plot (must match a name printed in the cell above)
ROI_NAME = "Belém"

QL_ASSET_KEY = "enclosure_ql_pauli_png"  # quicklook_jpg, enclosure_coh_ql_png, ...
SCENE_FIG_BASE_INCHES = 12  # longer side of each per-scene figure
PRODUCT_SLICE = slice(None)  # e.g. slice(0, 3) to test; slice(None) = all products

roi_names = [r["name"] for r in data["rois"]]
if ROI_NAME not in roi_names:
    raise ValueError(f"Unknown ROI_NAME={ROI_NAME!r}. Choose from: {roi_names}")

roi = next(r for r in data["rois"] if r["name"] == ROI_NAME)
products = sorted(roi["products"], key=lambda p: p.get("datetime") or "")
print(f"{ROI_NAME}: {len(products)} products")
if products:
    print(f"  {products[0]['datetime']} … {products[-1]['datetime']}")

# Figures go to disk (gitignored). Do not plt.show() — Jupyter would embed the PNGs.
import re

plt.ioff()
_roi_slug = re.sub(r"[^\w.-]+", "_", ROI_NAME).strip("_")
FIGURES_DIR = Path(topic_dir) / "figures" / _roi_slug
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


def save_figure(fig, stem: str) -> Path:
    path = FIGURES_DIR / f"{stem}.png"
    fig.savefig(path, dpi=120, bbox_inches="tight")
    plt.close(fig)
    print(f"saved {path.relative_to(topic_dir)}")
    return path


def scene_stem(index: int, product: dict) -> str:
    stamp = (product.get("datetime") or "unknown")[:19].replace("-", "").replace(":", "")
    return f"scene_{index:03d}_{stamp}"


print(f"Figures: {FIGURES_DIR.relative_to(topic_dir)}")


In [ ]:
from matplotlib.patches import Rectangle

from georef_quicklook import (
    fetch_image_array,
    load_product_quad,
    quad_bounds,
    warp_quicklook_to_view,
)

_quad_cache = {}


def add_roi_patch(ax, roi_bbox, **kwargs):
    defaults = {"fill": False, "edgecolor": "crimson", "linewidth": 2, "label": "ROI"}
    defaults.update(kwargs)
    ax.add_patch(
        Rectangle(
            (roi_bbox["west"], roi_bbox["south"]),
            roi_bbox["east"] - roi_bbox["west"],
            roi_bbox["north"] - roi_bbox["south"],
            **defaults,
        )
    )


def scene_figsize(bounds, base_inches: float):
    w, s, e, n = bounds
    lon_span = max(float(e - w), 1e-6)
    lat_span = max(float(n - s), 1e-6)
    ratio = lon_span / lat_span
    max_ratio = 4.0
    ratio = min(max(ratio, 1 / max_ratio), max_ratio)
    if ratio >= 1:
        return (base_inches * ratio, base_inches)
    return (base_inches, base_inches / ratio)


to_show = products[PRODUCT_SLICE]
if not to_show:
    print("No products for this ROI — pick another ROI_NAME or re-run the STAC search.")
else:
    roi_bb = roi["bbox"]

    fig, ax = plt.subplots(figsize=(9, 9))
    add_roi_patch(ax, roi_bb)
    for p in to_show:
        w, s, e, n = p["bbox"]
        ax.add_patch(Rectangle((w, s), e - w, n - s, fill=False, edgecolor="steelblue", alpha=0.5))
    pad = 0.05
    xs = [roi_bb["west"], roi_bb["east"]] + [p["bbox"][0] for p in to_show] + [p["bbox"][2] for p in to_show]
    ys = [roi_bb["south"], roi_bb["north"]] + [p["bbox"][1] for p in to_show] + [p["bbox"][3] for p in to_show]
    ax.set_xlim(min(xs) - pad, max(xs) + pad)
    ax.set_ylim(min(ys) - pad, max(ys) + pad)
    ax.set_xlabel("longitude")
    ax.set_ylabel("latitude")
    ax.set_title(f"{ROI_NAME}: ROI vs {len(to_show)} scene extents")
    ax.set_aspect("equal", adjustable="box")
    ax.legend(loc="upper right")
    plt.tight_layout()
    save_figure(fig, "scene_extents")

    print(f"Plotting {len(to_show)} scenes (KML georef; morning/evening row fix)...")
    for index, product in enumerate(to_show, start=1):
        url = product["assets"].get(QL_ASSET_KEY)
        figsize = scene_figsize(product["bbox"], SCENE_FIG_BASE_INCHES)
        fig, ax = plt.subplots(figsize=figsize)

        if not url:
            ax.set_title(f"[{index}/{len(to_show)}] missing {QL_ASSET_KEY}")
            ax.axis("off")
            plt.tight_layout()
            save_figure(fig, scene_stem(index, product))
            continue

        quad = load_product_quad(product, access_token, _quad_cache)
        west, south, east, north = quad_bounds(quad)
        pad_lon = max((east - west) * 0.01, 1e-5)
        pad_lat = max((north - south) * 0.01, 1e-5)
        view = (west - pad_lon, south - pad_lat, east + pad_lon, north + pad_lat)

        img = fetch_image_array(url, access_token)
        warped = warp_quicklook_to_view(
            img, quad, view, out_shape=(720, 520), product=product
        )
        ax.imshow(
            warped,
            extent=(view[0], view[2], view[1], view[3]),
            origin="upper",
            interpolation="nearest",
        )
        add_roi_patch(ax, roi_bb)
        ax.set_xlim(view[0], view[2])
        ax.set_ylim(view[1], view[3])
        ax.set_xlabel("longitude")
        ax.set_ylabel("latitude")
        ax.set_aspect("equal", adjustable="box")
        short_id = product["id"].split("_")[-1] if "_" in product["id"] else product["id"][:24]
        ax.set_title(
            f"[{index}/{len(to_show)}] {product['datetime'][:19]}Z | "
            f"ROI overlap {product['roi_overlap_fraction']:.1%} | {short_id}",
            fontsize=11,
        )
        plt.tight_layout()
        save_figure(fig, scene_stem(index, product))


In [ ]:
from matplotlib.patches import Rectangle

from georef_quicklook import (
    common_footprint_bounds,
    fetch_image_array,
    load_product_quad,
    preview_row_mapping_flipped,
    warp_quicklook_to_view,
)

# Common footprint gallery (KML quads). Filter pass if mixing breaks overlap.
DATE_SORT = "asc"  # "asc" | "desc"
PASS_FILTER = "all"  # "all" | "evening" (>=18 UTC) | "morning" (<12 UTC)
NCOLS = 2
PANEL_WIDTH = 8
PANEL_HEIGHT = 6
META_LINES = 4
WARP_SHAPE = (520, 520)
DRAW_ROI_ON_PANELS = True

_quad_cache = {}


def pass_matches(product: dict) -> bool:
    if PASS_FILTER == "all":
        return True
    hour = int(product["datetime"][11:13])
    if PASS_FILTER == "evening":
        return hour >= 18
    if PASS_FILTER == "morning":
        return hour < 12
    raise ValueError(f"Unknown PASS_FILTER={PASS_FILTER!r}")


def product_metadata_lines(product, index, total):
    props = product.get("properties") or {}
    dt = product.get("datetime") or "?"
    flip = preview_row_mapping_flipped(product)
    lines = [
        f"#{index}/{total}  {dt}",
        f"row map {'morning' if flip else 'evening'} | ROI {product.get('roi_overlap_fraction', 0):.2%}",
        f"mode {props.get('sar:instrument_mode', '—')}  level {props.get('processing:level', '—')}",
    ]
    pid = product.get("id", "")
    lines.append(pid if len(pid) <= 72 else pid[:69] + "...")
    return "\n".join(lines[:META_LINES])


subset = [p for p in products if pass_matches(p)]
if not subset:
    print("No products after PASS_FILTER — change PASS_FILTER or ROI_NAME.")
else:
    print(f"Loading KML for {len(subset)} scenes (PASS_FILTER={PASS_FILTER})...")
    quads = [load_product_quad(p, access_token, _quad_cache) for p in subset]
    west, south, east, north = common_footprint_bounds(quads)
    pad_lon = max((east - west) * 0.02, 1e-5)
    pad_lat = max((north - south) * 0.02, 1e-5)
    view = (west - pad_lon, south - pad_lat, east + pad_lon, north + pad_lat)

    print(f"Common KML overlap: W {west:.6f} S {south:.6f} E {east:.6f} N {north:.6f}")

    sorted_products = sorted(
        subset,
        key=lambda p: p.get("datetime") or "",
        reverse=(DATE_SORT == "desc"),
    )
    n = len(sorted_products)
    nrows = (n + NCOLS - 1) // NCOLS
    fig, axes = plt.subplots(
        nrows,
        NCOLS,
        figsize=(PANEL_WIDTH * NCOLS, (PANEL_HEIGHT + 1.0) * nrows),
        sharex=True,
        sharey=True,
    )
    axes_flat = [axes] if n == 1 and NCOLS == 1 else list(axes.ravel())

    for idx, product in enumerate(sorted_products):
        ax = axes_flat[idx]
        ax.set_title(product_metadata_lines(product, idx + 1, n), fontsize=8, loc="left", ha="left")
        url = product["assets"].get(QL_ASSET_KEY)
        if not url:
            ax.text(0.5, 0.5, f"missing {QL_ASSET_KEY}", ha="center", va="center", transform=ax.transAxes)
            continue

        quad = load_product_quad(product, access_token, _quad_cache)
        img = fetch_image_array(url, access_token)
        warped = warp_quicklook_to_view(
            img, quad, view, out_shape=WARP_SHAPE, product=product
        )
        ax.imshow(
            warped,
            extent=(view[0], view[2], view[1], view[3]),
            origin="upper",
            interpolation="nearest",
        )
        if DRAW_ROI_ON_PANELS:
            rb = roi["bbox"]
            ax.add_patch(
                Rectangle(
                    (rb["west"], rb["south"]),
                    rb["east"] - rb["west"],
                    rb["north"] - rb["south"],
                    fill=False,
                    edgecolor="crimson",
                    linewidth=1.2,
                )
            )
        ax.set_xlim(view[0], view[2])
        ax.set_ylim(view[1], view[3])
        ax.set_aspect("equal", adjustable="box")
        ax.tick_params(labelsize=7)

    for ax in axes_flat[n:]:
        ax.axis("off")

    sort_label = "oldest → newest" if DATE_SORT == "asc" else "newest → oldest"
    fig.suptitle(
        f"{ROI_NAME}: common footprint ({n} scenes, {sort_label}, {PASS_FILTER})",
        fontsize=12,
        y=1.01,
    )
    plt.tight_layout()
    save_figure(fig, f"common_footprint_{PASS_FILTER}_{DATE_SORT}")


In [ ]:
# Cropped latitude window. Run the common-footprint cell above first (defines view, sorted_products).
LAT_NORTH = -1.75  # top of each panel
LAT_SOUTH = -2.3   # cut everything south of this

if "view" not in globals() or "sorted_products" not in globals():
    raise RuntimeError("Run the common-footprint gallery cell first.")
if not (LAT_SOUTH < LAT_NORTH):
    raise ValueError("LAT_SOUTH must be less than LAT_NORTH.")

west, _, east, _ = view
cropped = (west, LAT_SOUTH, east, LAT_NORTH)
print(
    f"Cropped window: W {west:.5f}  S {LAT_SOUTH:.5f}  E {east:.5f}  N {LAT_NORTH:.5f}"
)

n = len(sorted_products)
nrows = (n + NCOLS - 1) // NCOLS
fig, axes = plt.subplots(
    nrows,
    NCOLS,
    figsize=(PANEL_WIDTH * NCOLS, (PANEL_HEIGHT + 1.0) * nrows),
    sharex=True,
    sharey=True,
)
axes_flat = [axes] if n == 1 and NCOLS == 1 else list(axes.ravel())

for idx, product in enumerate(sorted_products):
    ax = axes_flat[idx]
    ax.set_title(product_metadata_lines(product, idx + 1, n), fontsize=8, loc="left", ha="left")
    url = product["assets"].get(QL_ASSET_KEY)
    if not url:
        ax.text(0.5, 0.5, f"missing {QL_ASSET_KEY}", ha="center", va="center", transform=ax.transAxes)
        continue

    quad = load_product_quad(product, access_token, _quad_cache)
    img = fetch_image_array(url, access_token)
    warped = warp_quicklook_to_view(img, quad, cropped, out_shape=WARP_SHAPE, product=product)
    ax.imshow(
        warped,
        extent=(cropped[0], cropped[2], cropped[1], cropped[3]),
        origin="upper",
        interpolation="nearest",
    )
    ax.set_xlim(cropped[0], cropped[2])
    ax.set_ylim(LAT_SOUTH, LAT_NORTH)
    ax.set_aspect("equal", adjustable="box")
    ax.tick_params(labelsize=7)

for ax in axes_flat[n:]:
    ax.axis("off")

fig.suptitle(
    f"{ROI_NAME}: lat {LAT_SOUTH} to {LAT_NORTH} ({n} scenes)",
    fontsize=12,
    y=1.01,
)
plt.tight_layout()
lat_stem = f"lat_{LAT_SOUTH}_{LAT_NORTH}".replace("-", "m").replace(".", "p")
save_figure(fig, lat_stem)


In [10]:
# Pauli moduli as float32 GeoTIFF (LZW), one file per scene.
# Band 1 (red)   |HH - VV|  double bounce
# Band 2 (green) |HV + VH|  volume
# Band 3 (blue)  |HH + VV|  surface (single / odd bounce)
# Values are the complex moduli, not the 8-bit quicklook PNG.
# They are bilinearly resampled from slant range onto EPSG:4326 for QGIS.
# Each scene downloads amplitude + phase (hundreds of MB). Test with PRODUCT_SLICE = slice(0, 1).

import re
import urllib.parse
import urllib.request
from pathlib import Path

import numpy as np
import rasterio
from rasterio.control import GroundControlPoint
from rasterio.crs import CRS
from rasterio.enums import Resampling
from rasterio.transform import from_bounds
from rasterio.warp import calculate_default_transform, reproject
from rasterio.windows import Window, transform as window_transform

from georef_quicklook import fetch_text

missing = [
    name
    for name in ("access_token", "products", "ROI_NAME", "topic_dir", "roi")
    if name not in globals()
]
if missing:
    raise RuntimeError("Run the STAC and ROI cells first. Missing: " + ", ".join(missing))

PAULI_NODATA = -9999.0
OUT_DIR = Path(topic_dir) / "pauli_tif" / ROI_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)

_ID_RE = re.compile(
    r"BIO_(S\d)_STA__1S_(\d{8}T\d{6})_\d{8}T\d{6}_T_(G\d+)_(M\d+)_(C\d+)_(T\d+)_(F\d+)"
)
_ORBIT_RE = re.compile(r"<orbitPass>\s*(Ascending|Descending)\s*</orbitPass>")
_KNOWN_POL = {"HH", "HV", "VH", "VV", "XX"}
_BAND_DESCRIPTIONS = (
    "|HH-VV| double bounce",
    "|HV+VH| volume",
    "|HH+VV| surface (single/odd bounce)",
)


def scene_name(product: dict, orbit: str) -> str:
    match = _ID_RE.search(product["id"])
    if not match:
        date = (product.get("datetime") or "unknown").replace(":", "").replace("-", "")[:15]
        return f"{date}_{orbit}_pauli.tif"
    swath, date, gcov, major, cycle, track, frame = match.groups()
    return f"{date}_{swath}_{gcov}_{major}_{cycle}_{track}_{frame}_{orbit}_pauli.tif"


def orbit_pass(product: dict) -> str:
    url = product["assets"].get("enclosure_annot_xml_1")
    if not url:
        raise KeyError(f"{product['id']} has no enclosure_annot_xml_1")
    match = _ORBIT_RE.search(fetch_text(url, access_token))
    if not match:
        raise ValueError(f"orbitPass missing in annotation for {product['id']}")
    return match.group(1).lower()


def download_tiff(url: str, dest: Path, retries: int = 4) -> None:
    """Full GET. Range reads of these COGs fail mid-tile (empty HTTP body)."""
    dest.parent.mkdir(parents=True, exist_ok=True)
    part = dest.with_suffix(dest.suffix + ".part")
    request = urllib.request.Request(
        url, headers={"Authorization": f"Bearer {access_token}"}
    )
    last_error = None
    for attempt in range(1, retries + 1):
        try:
            with urllib.request.urlopen(request, timeout=180) as response, part.open("wb") as out:
                expected = int(response.headers.get("Content-Length") or 0)
                written = 0
                next_report = 50 * 1024 * 1024
                while True:
                    chunk = response.read(8 * 1024 * 1024)
                    if not chunk:
                        break
                    out.write(chunk)
                    written += len(chunk)
                    if written >= next_report:
                        total = f" / {expected / 1e6:.0f} MB" if expected else " MB"
                        print(f"    {written / 1e6:.0f}{total}")
                        next_report += 50 * 1024 * 1024
            if written == 0 or (expected and written != expected):
                raise OSError(f"downloaded {written} bytes, expected {expected or 'unknown'}")
            part.replace(dest)
            return
        except Exception as exc:
            last_error = exc
            part.unlink(missing_ok=True)
            dest.unlink(missing_ok=True)
            print(f"    attempt {attempt}/{retries} failed: {exc}")
    raise RuntimeError(f"could not download {dest.name}") from last_error


def read_polarimetric(url: str):
    name = Path(urllib.parse.urlparse(url).path).name
    dest = OUT_DIR / "_download" / name
    last_error = None
    for attempt in range(1, 4):
        try:
            if not dest.is_file():
                download_tiff(url, dest)
            with rasterio.open(dest) as src:
                data = src.read().astype(np.float32, copy=False)
                gcps, gcp_crs = src.gcps
                tags = src.tags()
                band_tags = [src.tags(i) for i in range(1, src.count + 1)]
                descriptions = list(src.descriptions)
                nodata = src.nodata
            dest.unlink(missing_ok=True)
            return data, gcps, gcp_crs, tags, band_tags, descriptions, nodata
        except Exception as exc:
            last_error = exc
            dest.unlink(missing_ok=True)
            print(f"    local read attempt {attempt}/3 failed: {exc}")
    raise RuntimeError(f"could not read {name}") from last_error


def polarisation_names(tags, band_tags, descriptions, count: int) -> list[str]:
    sequence = tags.get("PolarisationsSequence") or tags.get("POLARISATIONSSEQUENCE")
    if sequence and len(sequence.split()) == count:
        return [name.upper() for name in sequence.split()]
    if descriptions and all(d and str(d).strip().upper() in _KNOWN_POL for d in descriptions):
        return [str(d).strip().upper() for d in descriptions]
    names = []
    for band_tag in band_tags:
        found = None
        for key, value in band_tag.items():
            if value and "polar" in key.lower():
                found = str(value).strip().upper()
                break
        names.append(found)
    if all(names) and len(names) == count:
        return names
    if count == 4:
        print("  polarisation metadata missing; using lexicographic HH HV VH VV")
        return ["HH", "HV", "VH", "VV"]
    raise ValueError(f"could not identify {count} polarisations: {band_tags}")


def split_channels(data, names, nodata):
    invalid = ~np.isfinite(data)
    if nodata is not None:
        invalid |= data == np.float32(nodata)
    return {name: (data[index], invalid[index]) for index, name in enumerate(names)}


def as_complex(amp, phase, amp_invalid, phase_invalid):
    z = amp * np.exp(1j * phase)
    z = np.asarray(z, dtype=np.complex64)
    z[amp_invalid | phase_invalid] = np.nan
    return z


def pauli_moduli(abs_channels, phase_channels) -> np.ndarray:
    def cpx(name):
        amp, amp_invalid = abs_channels[name]
        phase, phase_invalid = phase_channels[name]
        return as_complex(amp, phase, amp_invalid, phase_invalid)

    for name in ("HH", "VV"):
        if name not in abs_channels or name not in phase_channels:
            raise KeyError(f"missing {name}; have {sorted(abs_channels)}")
    hh = cpx("HH")
    vv = cpx("VV")
    if "HV" in abs_channels and "VH" in abs_channels:
        cross = cpx("HV") + cpx("VH")
    elif "HV" in abs_channels:
        print("  VH absent; green = |2*HV| (merged cross-pol)")
        cross = 2 * cpx("HV")
    elif "VH" in abs_channels:
        print("  HV absent; green = |2*VH| (merged cross-pol)")
        cross = 2 * cpx("VH")
    else:
        raise KeyError(f"no cross-polarisation in {sorted(abs_channels)}")

    moduli = np.stack(
        [np.abs(hh - vv), np.abs(cross), np.abs(hh + vv)],
        axis=0,
    ).astype(np.float32)
    moduli[~np.isfinite(moduli)] = np.float32(PAULI_NODATA)
    return moduli


def gcp_lonlat(gcps, product):
    """Return GCP tuples (lon, lat, z, col, row). Swap if the tie points are stored lat, lon."""
    west, south, east, north = product["bbox"][:4]
    xs = [g.x for g in gcps]
    ys = [g.y for g in gcps]

    def centre_error(values, lo, hi):
        return abs(((min(values) + max(values)) / 2) - ((lo + hi) / 2))

    as_lonlat = centre_error(xs, west, east) + centre_error(ys, south, north)
    as_latlon = centre_error(xs, south, north) + centre_error(ys, west, east)
    swapped = as_latlon + 1e-4 < as_lonlat
    if swapped:
        print("  GCP axes look like lat, lon; swapping to lon, lat")
    points = []
    for g in gcps:
        lon, lat = (g.y, g.x) if swapped else (g.x, g.y)
        points.append((float(lon), float(lat), float(g.z or 0.0), float(g.col), float(g.row)))
    return points


def write_pauli_geotiff(path: Path, pauli: np.ndarray, gcps, gcp_crs, product, bounds):
    if not gcps:
        raise ValueError(f"no ground control points; cannot georeference {path.name}")
    src_height, src_width = pauli.shape[1], pauli.shape[2]
    src_crs = gcp_crs if gcp_crs is not None else CRS.from_epsg(4326)
    dst_crs = CRS.from_epsg(4326)
    points = [
        GroundControlPoint(row=row, col=col, x=lon, y=lat, z=z)
        for lon, lat, z, col, row in gcp_lonlat(gcps, product)
    ]
    full_transform, full_width, full_height = calculate_default_transform(
        src_crs, dst_crs, src_width, src_height, gcps=points
    )
    if bounds is None:
        dst_transform, dst_width, dst_height = full_transform, full_width, full_height
    else:
        west, south, east, north = bounds
        xres = abs(full_transform.a)
        yres = abs(full_transform.e)
        if xres == 0 or yres == 0:
            raise ValueError(f"invalid GCP resolution for {path.name}")
        dst_width = max(1, int(round((east - west) / xres)))
        dst_height = max(1, int(round((north - south) / yres)))
        dst_transform = from_bounds(west, south, east, north, dst_width, dst_height)

    destination = np.full((3, dst_height, dst_width), PAULI_NODATA, dtype=np.float32)
    reproject(
        pauli,
        destination,
        gcps=points,
        src_crs=src_crs,
        src_nodata=PAULI_NODATA,
        dst_transform=dst_transform,
        dst_crs=dst_crs,
        dst_nodata=PAULI_NODATA,
        resampling=Resampling.bilinear,
        num_threads=4,
    )
    valid = np.any(np.isfinite(destination) & (destination != PAULI_NODATA), axis=0)
    if not np.any(valid):
        raise ValueError(f"warp produced only nodata for {path.name}")
    rows, cols = np.where(valid)
    r0, r1 = int(rows.min()), int(rows.max()) + 1
    c0, c1 = int(cols.min()), int(cols.max()) + 1
    destination = destination[:, r0:r1, c0:c1]
    dst_transform = window_transform(Window(c0, r0, c1 - c0, r1 - r0), dst_transform)
    dst_height, dst_width = destination.shape[1], destination.shape[2]
    # Longitude-first WKT. EPSG:4326 authority WKT is latitude-first and QGIS then misses the layer.
    dst_crs = CRS.from_proj4("+proj=longlat +datum=WGS84 +no_defs +type=crs")

    with rasterio.open(
        path,
        "w",
        driver="GTiff",
        height=dst_height,
        width=dst_width,
        count=3,
        dtype="float32",
        crs=dst_crs,
        transform=dst_transform,
        nodata=PAULI_NODATA,
        compress="lzw",
        tiled=True,
        interleave="band",
        bigtiff="IF_SAFER",
    ) as dst:
        dst.write(destination)
        for index, description in enumerate(_BAND_DESCRIPTIONS, start=1):
            dst.set_band_description(index, description)
        dst.update_tags(
            PAULI_RED=_BAND_DESCRIPTIONS[0],
            PAULI_GREEN=_BAND_DESCRIPTIONS[1],
            PAULI_BLUE=_BAND_DESCRIPTIONS[2],
            CROP_BOUNDS=",".join(f"{v:.6f}" for v in bounds) if bounds is not None else "",
        )


if "cropped" not in globals() or "sorted_products" not in globals():
    raise RuntimeError("Run the cropped latitude-window cell first.")
scenes = list(sorted_products)
# Same window as the gallery: west/east from the common footprint, LAT_SOUTH to LAT_NORTH.
bounds = tuple(cropped)
print(
    f"Crop: W {bounds[0]:.5f} S {bounds[1]:.5f} E {bounds[2]:.5f} N {bounds[3]:.5f} (EPSG:4326)"
)
print(f"Writing {len(scenes)} scene(s) to {OUT_DIR}")

for index, product in enumerate(scenes, start=1):
    orbit = orbit_pass(product)
    out_path = OUT_DIR / scene_name(product, orbit)
    print(f"[{index}/{len(scenes)}] {out_path.name}")
    crop_tag = ",".join(f"{v:.6f}" for v in bounds)
    if out_path.is_file():
        with rasterio.open(out_path) as existing:
            if existing.tags().get("CROP_BOUNDS") == crop_tag:
                print("  exists, skip")
                continue
        print("  replacing file from a different window")
    abs_url = product["assets"].get("enclosure_i_abs_tiff")
    phase_url = product["assets"].get("enclosure_i_phase_tiff")
    if not abs_url or not phase_url:
        print("  missing amplitude or phase asset, skip")
        continue

    print("  downloading amplitude")
    amplitude, gcps, gcp_crs, tags, band_tags, descriptions, nodata = read_polarimetric(abs_url)
    print("  downloading phase")
    phase, _, _, phase_tags, phase_band_tags, phase_descriptions, phase_nodata = read_polarimetric(phase_url)
    if amplitude.shape != phase.shape:
        raise ValueError(f"amplitude {amplitude.shape} != phase {phase.shape} for {product['id']}")

    names = polarisation_names(tags, band_tags, descriptions, amplitude.shape[0])
    phase_names = polarisation_names(phase_tags, phase_band_tags, phase_descriptions, phase.shape[0])
    if phase_names != names:
        raise ValueError(f"amplitude polarisations {names} != phase {phase_names}")
    print("  polarisations:", " ".join(names))

    moduli = pauli_moduli(
        split_channels(amplitude, names, nodata),
        split_channels(phase, names, phase_nodata if phase_nodata is not None else nodata),
    )
    del amplitude, phase
    for description, band in zip(_BAND_DESCRIPTIONS, moduli):
        valid = band != PAULI_NODATA
        if not np.any(valid):
            print(f"  {description}: all nodata")
            continue
        print(f"  {description}: min {band[valid].min():.4g}  max {band[valid].max():.4g}")

    write_pauli_geotiff(out_path, moduli, gcps, gcp_crs, product, bounds)
    print(f"  saved {out_path}  ({out_path.stat().st_size / 1e6:.1f} MB)")

print("QGIS: multiband colour, red=band 1, green=band 2, blue=band 3.")
print("The display stretch does not change the stored moduli.")


Crop: W -49.15147 S -2.30000 E -48.74499 N -1.75000 (EPSG:4326)
Writing 21 scene(s) to /home/georg/code/biomass-hackathon-2026/topics/11-B-FLOODED-FOREST/pauli_tif/Belém
[1/21] 20260816T090302_S3_G01_M04_C01_T020_F309_ascending_pauli.tif
  exists, skip
[2/21] 20260819T090304_S3_G01_M04_C02_T020_F309_ascending_pauli.tif
  exists, skip
[3/21] 20260822T090306_S3_G01_M04_C03_T020_F309_ascending_pauli.tif
  exists, skip
[4/21] 20260825T090308_S3_G01_M04_C04_T020_F309_ascending_pauli.tif
  exists, skip
[5/21] 20260828T090310_S3_G01_M04_C05_T020_F309_ascending_pauli.tif
  exists, skip
[6/21] 20260831T090312_S3_G01_M04_C06_T020_F309_ascending_pauli.tif
  exists, skip
[7/21] 20260903T090315_S3_G01_M04_C07_T020_F309_ascending_pauli.tif
  downloading amplitude
    59 / 235 MB
    109 / 235 MB
    159 / 235 MB
    210 / 235 MB
  downloading phase
    59 / 234 MB
    109 / 234 MB
    159 / 234 MB
    210 / 234 MB
  polarisations: HH HV VH VV
  |HH-VV| double bounce: min 0  max 28.18
  |HV+VH| volum